# Análisis Exploratorio y de Negocio: Brazilian E-Commerce (Olist)

## Datos Académicos
* **Materia:** Ciencia de Datos
* **Institución:** Instituto Tecnológico Superior de Alvarado (ITSAV)
* **Profesor:** I.S.C. Fernando Mayorga Guittins
* **Integrantes del Equipo A:**
  1. Hiromi Alejandra Mosqueda Salto (Líder Técnico)
  2. Christopher Vasquez Carvajal (Desarrollador - Limpieza)
  3. Erik Alberto Hernández Orozco(Desarrollador - Ingeniería de Datos)
  4. Alberto Quevedo Gapi (Desarrollador - EDA y Visualización)
* **Fecha:** Octubre 2026

## Contexto de los Datos
El conjunto de datos analizado pertenece a **Olist**, la plataforma de comercio electrónico más grande de Brasil. Contiene información real y anonimizada de aproximadamente **100,000 órdenes de compra** realizadas entre 2016 y 2018. El modelo relacional abarca múltiples dimensiones comerciales: estado del pedido, atributos físicos del producto, pagos, opiniones de clientes y tiempos logísticos de entrega.

## Objetivos y Preguntas de Negocio
El objetivo principal es transformar datos crudos en métricas accionables respondiendo a tres interrogantes estratégicas:
1. **Pregunta 1 (Satisfacción Logística):** ¿Cómo impacta el tiempo de entrega real frente al estimado en la calificación de satisfacción del cliente? *(Cruce de `olist_orders` con `olist_order_reviews`)*.
2. **Pregunta 2 (Rendimiento Comercial):** ¿Cuáles son las 5 categorías de productos que generan mayores ingresos económicos en la plataforma? *(Cruce de `olist_order_items`, `olist_products` y `product_category_name_translation`)*.
3. **Pregunta 3 (Análisis Financiero):** ¿Cuál es la relación entre el método de pago elegido por el usuario y el valor total del ticket de compra? *(Cruce de `olist_orders` con `olist_order_payments`)*.


In [2]:
# =====================================================================
# SECCIÓN 2: IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN GLOBAL
# =====================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

# Configuración global de Pandas para visualización de DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Configuración de estilo gráfico para Seaborn y Matplotlib
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

# Silenciar advertencias de ejecución
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías e importaciones configuradas exitosamente.")


✅ Librerías e importaciones configuradas exitosamente.


## Carga e Inspección Inicial de los Datos

En esta sección se realiza el montaje de Google Drive para acceder a la carpeta `Proyecto_Final/data/` y se cargan los 6 archivos CSV que conforman el ecosistema de Olist.


In [3]:
# =====================================================================
# SECCIÓN 3: MONTAJE Y CARGA DE DATANETS DE OLIST
# =====================================================================

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir ruta absoluta hacia la carpeta de datos
DATA_PATH = '/content/drive/MyDrive/Proyecto_Final/data/'

# 3. Cargar las 6 tablas principales del proyecto
df_orders = pd.read_csv(os.path.join(DATA_PATH, 'olist_orders_dataset.csv'))
df_order_items = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_items_dataset.csv'))
df_products = pd.read_csv(os.path.join(DATA_PATH, 'olist_products_dataset.csv'))
df_reviews = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_reviews_dataset.csv'))
df_payments = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_payments_dataset.csv'))
df_translation = pd.read_csv(os.path.join(DATA_PATH, 'product_category_name_translation.csv'))

print("✅ Todos los datasets fueron cargados correctamente.")


Mounted at /content/drive
✅ Todos los datasets fueron cargados correctamente.


In [4]:
# =====================================================================
# INSPECCIÓN ESTRUCTURAL INICIAL DE LAS TABLAS PRINCIPALES
# =====================================================================

print("=== 1. FORMA DE LOS DATASETS (Filas, Columnas) ===")
print(f"Orders:   {df_orders.shape}")
print(f"Items:    {df_order_items.shape}")
print(f"Products: {df_products.shape}")
print(f"Reviews:  {df_reviews.shape}")
print(f"Payments: {df_payments.shape}")

print("\n=== 2. VISTA PREVIA DE PEDIDOS (df_orders.head()) ===")
display(df_orders.head(3))

print("\n=== 3. INFORMACIÓN DE TIPOS Y NULOS EN PEDIDOS (df_orders.info()) ===")
df_orders.info()

print("\n=== 4. RESUMEN ESTADÍSTICO DE PAGOS (df_payments.describe()) ===")
display(df_payments.describe())


=== 1. FORMA DE LOS DATASETS (Filas, Columnas) ===
Orders:   (99441, 8)
Items:    (112650, 7)
Products: (32951, 9)
Reviews:  (99224, 7)
Payments: (103886, 5)

=== 2. VISTA PREVIA DE PEDIDOS (df_orders.head()) ===


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



=== 3. INFORMACIÓN DE TIPOS Y NULOS EN PEDIDOS (df_orders.info()) ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB

=== 4. RESUMEN ESTADÍSTICO DE PAGOS (df_payments.describe()) ===


,payment_sequential,payment_installments,payment_value
count,103886.00,103886.00,103886.00
mean,1.09,2.85,154.10
std,0.71,2.69,217.49
min,1.00,0.00,0.00
25%,1.00,1.00,56.79
50%,1.00,1.00,100.00
75%,1.00,4.00,171.84
max,29.00,24.00,13664.08
